# C1 BiLSTM Focused Hyperparameter Tuning

A **narrowed** random search over the BiLSTM C1 classifier (single vs pileup), built from the findings of the earlier broad search (`c1_tune.ipynb`). Prior top-10 configurations told us which hyperparameter values were worth re-sampling.

## What was narrowed and why

| Hyperparameter | Broad search | Focused search | Reason |
|---|---|---|---|
| `lstm_units` | 32, 64, 128 | **32, 64** | 128 only placed once (531k params, ~15× rank 1) |
| `n_lstm_layers` | 1, 2 | 1, 2 | both viable |
| `dense_units` | 16, 32, 64 | **16, 32** | 64 only appeared at ranks 5–10 |
| `dropout` | 0.1–0.4 | **0.1, 0.15, 0.2, 0.25, 0.3** | 0.4/0.5 failed catastrophically; finer resolution near winning region |
| `recurrent_dropout` | 0.0, 0.1, 0.2 | **0.0, 0.1** | no prior evidence to constrain; drop 0.2 for compute |
| `learning_rate` | 1e-4–2e-3 | **5e-4, 1e-3, 2e-3** | 1e-4 caused slow convergence, usually didn't finish |
| `batch_size` | 256, 512, 1024 | 256, 512, 1024 | all three viable |
| `optimizer` | adam, adamw, nadam | **adam** | no prior data on others; keep search tight |

**Search size:** 720 combinations (broad search had 3,456).  
**Trials:** 25 (broad search had 50). **Expected compute: ~50% of the broad search.**

All other safeguards are identical to the broad version: per-trial overfit detection, val-only ranking, held-out test-set evaluation of the top 5 finalists, and top-3 training-curve plots.

In [1]:
import itertools
import random
import time

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print(f"TensorFlow {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

I0000 00:00:1775855593.352198 1198428 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1775855593.983446 1198428 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow 2.22.0-dev0+selfbuilt
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Load and split data

Singles come from `processed_waveforms.npz`, restricted to `clean_indices` (the singles never used in pileup generation). Pileups come from `pileup_waveforms.npz`. We balance the two classes, Euclidean-normalize, and split 60/20/20.

In [2]:
singles = np.load("processed_waveforms.npz")
X_singles_all = singles["X_voltage"].astype(np.float32)

pileups = np.load("pileup_waveforms.npz")
X_pileup = pileups["pileup_wf"].astype(np.float32)
n_pileup = X_pileup.shape[0]

clean_idx = pileups["clean_indices"]
X_singles = X_singles_all[clean_idx]
n_singles = X_singles.shape[0]

rng = np.random.default_rng(42)
n_balanced = min(n_singles, n_pileup)
single_idx = rng.choice(n_singles, size=n_balanced, replace=False)
pileup_idx = rng.choice(n_pileup, size=n_balanced, replace=False)

X_all = np.concatenate([X_singles[single_idx], X_pileup[pileup_idx]], axis=0)
y_all = np.concatenate([
    np.zeros(n_balanced, dtype=np.int64),  # 0 = single
    np.ones(n_balanced, dtype=np.int64),   # 1 = pileup
])

# Euclidean-normalize each waveform
norms = np.linalg.norm(X_all, axis=1, keepdims=True)
norms[norms == 0] = 1.0
X_all = X_all / norms

# Shuffle
shuffle = rng.permutation(len(y_all))
X_all = X_all[shuffle]
y_all = y_all[shuffle]

# 60/20/20 train/val/test split
X_tv, X_test, y_tv, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)

# Keras LSTM expects (batch, timesteps, features)
X_train = X_train[..., np.newaxis]
X_val   = X_val[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Class balance (train): single={(y_train==0).sum():,}  pileup={(y_train==1).sum():,}")

Train: (208002, 104, 1)  Val: (69335, 104, 1)  Test: (69335, 104, 1)
Class balance (train): single=104,001  pileup=104,001


## Search space

In [3]:
SEARCH_SPACE = {
    "lstm_units":        [32, 64],               # dropped 128 (too big for marginal gain)
    "n_lstm_layers":     [1, 2],
    "dense_units":       [16, 32],               # dropped 64
    "dropout":           [0.1, 0.15, 0.2, 0.25, 0.3],  # finer resolution, dropped 0.4/0.5
    "recurrent_dropout": [0.0],             # forced to 0 so cuDNN fast-path is used
    "learning_rate":     [5e-4, 1e-3, 2e-3],     # dropped 1e-4
    "batch_size":        [256, 512, 1024],
    "optimizer":         ["adam"],               # constrain to known-good
}

N_TRIALS   = 25            # half the broad-search budget
MAX_EPOCHS = 25
PATIENCE   = 6

# Maximum train_acc - val_acc before flagging a trial as overfit
OVERFIT_GAP_THRESHOLD = 0.05

total_combos = 1
for v in SEARCH_SPACE.values():
    total_combos *= len(v)
print(f"Total possible combos: {total_combos:,}")
print(f"Sampling {N_TRIALS} random configurations")

Total possible combos: 360
Sampling 25 random configurations


## Model builder

In [4]:
def get_optimizer(name, learning_rate):
    if name == "adam":  return keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "adamw": return keras.optimizers.AdamW(learning_rate=learning_rate)
    if name == "nadam": return keras.optimizers.Nadam(learning_rate=learning_rate)
    raise ValueError(f"Unknown optimizer: {name}")


def build_bilstm(hp, n_samples=104):
    model_layers = [keras.layers.Input(shape=(n_samples, 1))]
    for i in range(hp["n_lstm_layers"]):
        return_seq = (i < hp["n_lstm_layers"] - 1)
        model_layers.append(
            layers.Bidirectional(
                layers.LSTM(
                    hp["lstm_units"],
                    return_sequences=return_seq,
                    dropout=hp["dropout"],
                    recurrent_dropout=hp["recurrent_dropout"],
                )
            )
        )
    model_layers.append(layers.Dense(hp["dense_units"], activation="relu"))
    model_layers.append(layers.Dropout(hp["dropout"]))
    model_layers.append(layers.Dense(1, activation="sigmoid"))

    model = keras.Sequential(model_layers)
    model.compile(
        optimizer=get_optimizer(hp["optimizer"], hp["learning_rate"]),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

## Trial evaluation

Each trial trains a model on the train split, with early stopping monitored on `val_loss`. After training:

1. Re-measure train and val accuracy/loss
2. Compute `overfit_gap = train_acc - val_acc`
3. Compute `composite_score = 0.5 * val_loss + 0.5 * (1 - val_acc)` (val-only)
4. Save the full training history so we can plot loss curves later

The test set is NOT touched at this stage.

In [5]:
def composite_score(val_loss, val_acc, alpha=0.5):
    return alpha * val_loss + (1 - alpha) * (1 - val_acc)


def run_trial(hp, X_train, X_val, y_train, y_val,
              max_epochs=MAX_EPOCHS, patience=PATIENCE):
    keras.backend.clear_session()
    model = build_bilstm(hp, n_samples=X_train.shape[1])
    n_params = model.count_params()

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience,
            restore_best_weights=True, verbose=0,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", patience=3, factor=0.5, verbose=0,
        ),
    ]

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=max_epochs,
        batch_size=hp["batch_size"],
        callbacks=callbacks,
        verbose=0,
    )
    train_time = time.time() - t0

    # Re-measure on train + val after early stopping has restored best weights
    train_loss, train_acc = model.evaluate(X_train, y_train, batch_size=hp["batch_size"], verbose=0)
    val_loss,   val_acc   = model.evaluate(X_val,   y_val,   batch_size=hp["batch_size"], verbose=0)

    overfit_gap = train_acc - val_acc
    is_overfit = overfit_gap > OVERFIT_GAP_THRESHOLD
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

    return {
        **hp,
        "n_params": n_params,
        "best_epoch": best_epoch,
        "train_loss": float(train_loss),
        "train_acc": float(train_acc),
        "val_loss": float(val_loss),
        "val_acc": float(val_acc),
        "overfit_gap": float(overfit_gap),
        "is_overfit": bool(is_overfit),
        "composite_score": float(composite_score(val_loss, val_acc)),
        "train_time_s": round(train_time, 1),
    }, history.history

## Run the focused random search

Histories are kept in memory (a list of dicts) so we can plot loss curves for the top 3 later. The summary results are saved incrementally to `c1_tune_focused_results.csv`.

In [6]:
random.seed(42)
all_combos = list(itertools.product(*SEARCH_SPACE.values()))
keys = list(SEARCH_SPACE.keys())
sampled = random.sample(all_combos, min(N_TRIALS, len(all_combos)))

results = []
histories = []  # parallel list, same order as results

for trial_num, combo in enumerate(sampled):
    hp = dict(zip(keys, combo))
    print(f"\n{'='*78}")
    print(f"Trial {trial_num+1}/{len(sampled)}")
    print(f"  arch: lstm={hp['lstm_units']} layers={hp['n_lstm_layers']} dense={hp['dense_units']} "
          f"drop={hp['dropout']} rdrop={hp['recurrent_dropout']}")
    print(f"  opt:  {hp['optimizer']} lr={hp['learning_rate']} bs={hp['batch_size']}")

    try:
        result, history = run_trial(hp, X_train, X_val, y_train, y_val)
        results.append(result)
        histories.append(history)
        flag = " [OVERFIT]" if result["is_overfit"] else ""
        print(f"  -> train_acc={result['train_acc']:.4f}  val_acc={result['val_acc']:.4f}  "
              f"gap={result['overfit_gap']:+.4f}{flag}  "
              f"score={result['composite_score']:.5f}  "
              f"params={result['n_params']:,}  time={result['train_time_s']}s")
    except Exception as e:
        print(f"  -> FAILED: {e}")
        continue

    # Incremental save (summary only — histories live in memory)
    pd.DataFrame(results).to_csv("c1_tune_focused_results.csv", index=False)

n_overfit = sum(1 for r in results if r['is_overfit'])
print(f"\n{'='*78}")
print(f"Complete: {len(results)}/{len(sampled)} successful, {n_overfit} flagged as overfit")


Trial 1/25
  arch: lstm=64 layers=2 dense=32 drop=0.15 rdrop=0.0
  opt:  adam lr=0.001 bs=256


I0000 00:00:1775855596.018173 1198428 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12442 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0a


I0000 00:00:1775855598.365471 1198606 cuda_dnn.cc:461] Loaded cuDNN version 91900


  -> train_acc=0.9743  val_acc=0.9743  gap=-0.0000  score=0.05257  params=136,769  time=289.3s

Trial 2/25
  arch: lstm=32 layers=1 dense=32 drop=0.15 rdrop=0.0
  opt:  adam lr=0.001 bs=256


  -> train_acc=0.9379  val_acc=0.9380  gap=-0.0001  score=0.11704  params=10,817  time=98.4s

Trial 3/25
  arch: lstm=32 layers=1 dense=16 drop=0.15 rdrop=0.0
  opt:  adam lr=0.001 bs=256


...

  -> train_acc=0.9502  val_acc=0.9496  gap=+0.0005  score=0.09394  params=9,761  time=51.2s

Trial 22/25
  arch: lstm=64 layers=2 dense=16 drop=0.15 rdrop=0.0
  opt:  adam lr=0.002 bs=1024


  -> train_acc=0.9714  val_acc=0.9713  gap=+0.0001  score=0.05712  params=134,689  time=127.7s

Trial 23/25
  arch: lstm=32 layers=2 dense=16 drop=0.15 rdrop=0.0
  opt:  adam lr=0.0005 bs=1024


  -> train_acc=0.9375  val_acc=0.9371  gap=+0.0004  score=0.11315  params=34,593  time=39.4s

Trial 24/25
  arch: lstm=64 layers=2 dense=32 drop=0.15 rdrop=0.0
  opt:  adam lr=0.002 bs=1024


  -> train_acc=0.9608  val_acc=0.9609  gap=-0.0001  score=0.07429  params=136,769  time=87.5s

Trial 25/25
  arch: lstm=64 layers=2 dense=32 drop=0.3 rdrop=0.0
  opt:  adam lr=0.002 bs=1024


  -> train_acc=0.9575  val_acc=0.9577  gap=-0.0002  score=0.07830  params=136,769  time=87.3s

Complete: 25/25 successful, 0 flagged as overfit


## Top configurations (overfit excluded)

In [7]:
df = pd.DataFrame(results)
print(f"Total trials: {len(df)}")
print(f"Overfit trials excluded: {df['is_overfit'].sum()}")
print(f"Eligible trials: {(~df['is_overfit']).sum()}")
print()

clean_df = df[~df["is_overfit"]].copy()

print("=" * 100)
print("TOP 10 BY COMPOSITE SCORE (overfit excluded)")
print("=" * 100)
cols = ["lstm_units", "n_lstm_layers", "dense_units", "dropout",
        "recurrent_dropout", "learning_rate", "batch_size", "optimizer",
        "train_acc", "val_acc", "overfit_gap", "composite_score", "n_params", "best_epoch"]
print(clean_df.sort_values("composite_score").head(10)[cols].to_string())

print("\n\nFull comparison: clean vs overfit")
print("=" * 60)
if df['is_overfit'].any():
    summary = df.groupby("is_overfit").agg(
        n=("composite_score", "count"),
        mean_train_acc=("train_acc", "mean"),
        mean_val_acc=("val_acc", "mean"),
        mean_overfit_gap=("overfit_gap", "mean"),
        best_score=("composite_score", "min"),
    )
    print(summary.round(4).to_string())

Total trials: 25
Overfit trials excluded: 0
Eligible trials: 25

TOP 10 BY COMPOSITE SCORE (overfit excluded)
    lstm_units  n_lstm_layers  dense_units  dropout  recurrent_dropout  learning_rate  batch_size optimizer  train_acc   val_acc  overfit_gap  composite_score  n_params  best_epoch
0           64              2           32     0.15                0.0          0.001         256      adam   0.974294  0.974328    -0.000034         0.052574    136769          21
21          64              2           16     0.15                0.0          0.002        1024      adam   0.971366  0.971299     0.000067         0.057119    134689          22
5           32              2           16     0.20                0.0          0.002         256      adam   0.968880  0.968169     0.000711         0.061750     34593          14
23          64              2           32     0.15                0.0          0.002        1024      adam   0.960798  0.960929    -0.000130         0.074288    1367

## Per-hyperparameter effects

How each hyperparameter individually affects validation accuracy. Red dots = mean for that value.

In [ ]:
import os; os.makedirs("figures", exist_ok=True)
hp_to_plot = list(SEARCH_SPACE.keys())
ncols = 4
nrows = int(np.ceil(len(hp_to_plot) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = axes.flatten()

for ax, hp_name in zip(axes, hp_to_plot):
    vals = clean_df[hp_name].astype(str)
    unique = sorted(vals.unique())
    for v in unique:
        mask = vals == v
        ax.scatter([v] * mask.sum(),
                   clean_df.loc[mask, "val_acc"].values * 100,
                   alpha=0.5, s=30, color="tab:blue")
    means = clean_df.groupby(vals)["val_acc"].mean() * 100
    ax.scatter(means.index, means.values, color="tab:red", s=80, zorder=5, label="Mean")
    ax.set_title(hp_name, fontsize=10)
    ax.set_ylabel("Val accuracy [%]", fontsize=8)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.grid(True, alpha=0.3)

for j in range(len(hp_to_plot), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Per-hyperparameter effect on validation accuracy (overfit excluded)", fontsize=13)
plt.tight_layout()
plt.savefig("figures/c1_tune_focused_hyperparameter_effects.png", dpi=120, bbox_inches="tight")
plt.show()

## Training curves for the top 3 finalists

These are the configurations that placed best on the val-only composite score (after excluding overfit trials). Each row shows loss and accuracy over epochs for one finalist. Big train/val gaps would indicate residual overfitting that the gap check missed.

In [ ]:
# Map original index in `results` to the clean rank
clean_df = df[~df["is_overfit"]].copy()
top3 = clean_df.sort_values("composite_score").head(3)

fig, axes = plt.subplots(3, 2, figsize=(12, 10))

for plot_row, (orig_idx, row) in enumerate(top3.iterrows()):
    history = histories[orig_idx]

    title = (f"#{plot_row+1}  lstm={int(row.lstm_units)} layers={int(row.n_lstm_layers)} "
             f"dense={int(row.dense_units)} drop={row.dropout} "
             f"lr={row.learning_rate} bs={int(row.batch_size)} {row.optimizer}\n"
             f"val_acc={row.val_acc:.4f}  gap={row.overfit_gap:+.4f}  score={row.composite_score:.5f}")

    # Loss curves
    ax = axes[plot_row, 0]
    epochs = np.arange(1, len(history["loss"]) + 1)
    ax.plot(epochs, history["loss"],     label="train", color="tab:blue")
    ax.plot(epochs, history["val_loss"], label="val",   color="tab:orange")
    ax.axvline(row.best_epoch, color="green", linestyle="--", alpha=0.5, label=f"best epoch ({int(row.best_epoch)})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (binary crossentropy)")
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Accuracy curves
    ax = axes[plot_row, 1]
    ax.plot(epochs, np.array(history["accuracy"])     * 100, label="train", color="tab:blue")
    ax.plot(epochs, np.array(history["val_accuracy"]) * 100, label="val",   color="tab:orange")
    ax.axvline(row.best_epoch, color="green", linestyle="--", alpha=0.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy [%]")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Top 3 finalists — training curves", fontsize=13)
plt.tight_layout()
plt.savefig("figures/c1_tune_focused_top3_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## Honest test-set evaluation of top 5 finalists

Re-train each of the top 5 configurations from scratch and evaluate them once on the held-out test set. The `val → test gap` column shows how much the val-based composite score over- or under-estimated the true performance — a large positive gap is a sign of selection bias.

In [10]:
top5 = clean_df.sort_values("composite_score").head(5)
final_results = []

for rank, (_, row) in enumerate(top5.iterrows(), 1):
    hp = {k: row[k] for k in SEARCH_SPACE.keys()}
    # Cast back to native types
    hp["lstm_units"]    = int(hp["lstm_units"])
    hp["n_lstm_layers"] = int(hp["n_lstm_layers"])
    hp["dense_units"]   = int(hp["dense_units"])
    hp["batch_size"]    = int(hp["batch_size"])
    hp["dropout"]       = float(hp["dropout"])
    hp["recurrent_dropout"] = float(hp["recurrent_dropout"])
    hp["learning_rate"] = float(hp["learning_rate"])

    print(f"\nFinalist {rank}: lstm={hp['lstm_units']} layers={hp['n_lstm_layers']} "
          f"dense={hp['dense_units']} drop={hp['dropout']} lr={hp['learning_rate']} "
          f"bs={hp['batch_size']} {hp['optimizer']}")
    print(f"  val_acc (search) = {row.val_acc:.4f}")

    keras.backend.clear_session()
    model = build_bilstm(hp, n_samples=X_train.shape[1])
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, verbose=0),
    ]
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=MAX_EPOCHS, batch_size=hp["batch_size"],
              callbacks=callbacks, verbose=0)

    test_loss, test_acc = model.evaluate(X_test, y_test, batch_size=hp["batch_size"], verbose=0)
    val_test_gap = test_acc - row.val_acc

    print(f"  test_acc         = {test_acc:.4f}")
    print(f"  val -> test gap   = {val_test_gap:+.4f}")

    final_results.append({
        "rank": rank,
        "lstm_units": hp["lstm_units"],
        "n_lstm_layers": hp["n_lstm_layers"],
        "dense_units": hp["dense_units"],
        "dropout": hp["dropout"],
        "recurrent_dropout": hp["recurrent_dropout"],
        "learning_rate": hp["learning_rate"],
        "batch_size": hp["batch_size"],
        "optimizer": hp["optimizer"],
        "val_acc": row.val_acc,
        "test_acc": float(test_acc),
        "val_test_gap": float(val_test_gap),
    })

print("\n" + "=" * 70)
print("FINALIST SUMMARY")
print("=" * 70)
final_df = pd.DataFrame(final_results)
print(final_df.to_string(index=False))

df.to_csv("c1_tune_focused_results.csv", index=False)
final_df.to_csv("c1_tune_focused_finalists.csv", index=False)
print("\nSaved c1_tune_focused_results.csv and c1_tune_focused_finalists.csv")


Finalist 1: lstm=64 layers=2 dense=32 drop=0.15 lr=0.001 bs=256 adam
  val_acc (search) = 0.9743


  test_acc         = 0.9723
  val -> test gap   = -0.0020

Finalist 2: lstm=64 layers=2 dense=16 drop=0.15 lr=0.002 bs=1024 adam
  val_acc (search) = 0.9713


  test_acc         = 0.9471
  val -> test gap   = -0.0242

Finalist 3: lstm=32 layers=2 dense=16 drop=0.2 lr=0.002 bs=256 adam
  val_acc (search) = 0.9682


  test_acc         = 0.9690
  val -> test gap   = +0.0008

Finalist 4: lstm=64 layers=2 dense=32 drop=0.15 lr=0.002 bs=1024 adam
  val_acc (search) = 0.9609


  test_acc         = 0.9261
  val -> test gap   = -0.0348

Finalist 5: lstm=64 layers=2 dense=32 drop=0.3 lr=0.002 bs=1024 adam
  val_acc (search) = 0.9577


  test_acc         = 0.9534
  val -> test gap   = -0.0043

FINALIST SUMMARY
 rank  lstm_units  n_lstm_layers  dense_units  dropout  recurrent_dropout  learning_rate  batch_size optimizer  val_acc  test_acc  val_test_gap
    1          64              2           32     0.15                0.0          0.001         256      adam 0.974328  0.972308     -0.002019
    2          64              2           16     0.15                0.0          0.002        1024      adam 0.971299  0.947141     -0.024158
    3          32              2           16     0.20                0.0          0.002         256      adam 0.968169  0.968977      0.000808
    4          64              2           32     0.15                0.0          0.002        1024      adam 0.960929  0.926141     -0.034788
    5          64              2           32     0.30                0.0          0.002        1024      adam 0.957727  0.953443     -0.004284

Saved c1_tune_focused_results.csv and c1_tune_focused_final